In [ ]:
#@title ⚙️ نصب و دانلود
import base64
encoded_text = "Q3JlYXRlIGJ5IDogYWlnb2xkZW4="
decoded_text = base64.b64decode(encoded_text.encode()).decode()
print(decoded_text)
print('='*30)

!pip install -q --upgrade gemini-srt-translator
!pip install -q -U google-genai yt-dlp pysrt pydub
!pip install -q omnivoice
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg rubberband-cli

import torch
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

print('در حال دانلود و لود مدل ...')
omni_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ همه چیز آماده‌ست، مدل لود شد!')


In [ ]:
#@title 🔑 تنظیمات کلی پروژه
import os

#@markdown ##### 🔑 یک کلید دسترسی دریافت کنید
#@markdown ##### https://aistudio.google.com/api-keys
GOOGLE_API_KEY = '' #@param {type:"string"}
#@markdown ---
#@markdown #### 🌎 زبان مقصد برای ترجمه
Target_Language = 'es' #@param ["fa", "en", "ar", "de", "fr", "es", "it", "tr", "ru", "zh-Hans", "ja", "ko", "hi", "pl", "pt-BR"]
#@markdown ---
#@markdown #### ✨ لحن گویندگی (برای ترجمه و لحن متن)
Tone = 'عامیانه / گفتاری' #@param ["عامیانه / گفتاری", "رسمی / اخبار / کتابی", "آموزشی / آکادمیک", "پرانرژی / تبلیغاتی", "طبیعی / پیش‌فرض"]
#@markdown ---
#@markdown #### 🎚️ صدای اصلی ویدیو
Keep_Original_Audio = False #@param {type:"boolean"}
Original_Audio_Volume = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown ---
#@markdown #### ⏱️ زمان‌بندی و سینک
Allow_Timeline_Stretch = False #@param {type:"boolean"}
Max_Stretch_Percent = 10 #@param {type:"slider", min:0, max:25, step:1}
Balance_Speed_Across_Segments = True #@param {type:"boolean"}
Max_Speed_Factor = 1.4 #@param {type:"slider", min:1.0, max:2.0, step:0.1}

# مقادیر ثابت OmniVoice (بدون کنترل روی صفحه)
OmniVoice_Steps = 32
OmniVoice_Speed = 1.0

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print('✅ تنظیمات ثبت شد')
print(f'🌐 زبان مقصد: {Target_Language}')
print(f'🎭 لحن بیان: {Tone}')
print(f'🎚️ صدای اصلی: {"حفظ شود" if Keep_Original_Audio else "حذف شود"} (حجم {Original_Audio_Volume})')
print(f'⏱️ کشش زمان‌بندی: {"فعال" if Allow_Timeline_Stretch else "غیرفعال"} ({Max_Stretch_Percent}%)')
print(f'⚡ تعادل سرعت: {"فعال" if Balance_Speed_Across_Segments else "غیرفعال"} (حداکثر {Max_Speed_Factor}x)')


In [ ]:
#@title ####📂 **Select video**
import subprocess, os, glob, shutil
from google.colab import files

for f in (glob.glob('input_video*') + glob.glob('yt_audio*') +
          ['audio.mp3', 'audio.srt', 'audio_translated.srt',
           'dubbed_audio.mp3', 'final_dubbed_video.mp4', 'raw_segments_combined.mp3']):
    if os.path.exists(f):
        os.remove(f)
if os.path.exists('dubbing_project'):
    shutil.rmtree('dubbing_project')
if os.path.exists('audio_chunks'):
    shutil.rmtree('audio_chunks')

Upload_Method = 'فایل از دستگاه' #@param ["یوتیوب", "فایل از دستگاه"]
YT_Link = '' #@param {type:"string"}

AUDIO_FILE = None
VIDEO_FILE = None
SRT_FILE = None
SUBTITLE_READY = False

# پارامترهای ضد بوت برای yt-dlp
YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=android,mweb,ios']

if Upload_Method == 'فایل از دستگاه':
    print('📁 فایل ویدیو یا صدا را آپلود کنید:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('❌ فایلی آپلود نشد!')
    fname = list(uploaded.keys())[0]
    ext = fname.rsplit('.', 1)[-1].lower()
    if ext in ['mp4', 'mkv', 'avi', 'mov', 'webm']:
        VIDEO_FILE = 'input_video.mp4'
        os.rename(fname, VIDEO_FILE)
        res = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_FILE, '-vn', '-acodec', 'libmp3lame',
                              '-ab', '128k', '-ac', '1', 'audio.mp3'], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ استخراج صدا از فایل آپلودی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = 'audio.mp3'
        print('✅ ویدیو آپلود شد، صدا استخراج شد')
    else:
        mp3_name = 'audio.mp3'
        res = subprocess.run(['ffmpeg', '-y', '-i', fname, '-acodec', 'libmp3lame',
                              '-ab', '128k', '-ac', '1', mp3_name], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ تبدیل فایل صوتی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = mp3_name
        print('✅ فایل صوتی آماده شد')

else:
    if not YT_Link.strip():
        raise ValueError('❌ لینک یوتیوب وارد نشده است!')

    print('🎬 در حال دانلود مستقیم ویدیو از یوتیوب...')
    res_vid = subprocess.run(['yt-dlp', '-f', 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                              '-o', 'input_video.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
    if res_vid.returncode != 0:
        # تلاش مجدد با کلاینت جایگزین در صورت مسدودی
        YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=tv,mweb']
        res_vid = subprocess.run(['yt-dlp', '-f', 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                                  '-o', 'input_video.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
        if res_vid.returncode != 0:
            raise RuntimeError(f'❌ دانلود ویدیو از یوتیوب ناموفق بود:\n{res_vid.stderr}')

    vfiles = glob.glob('input_video.*')
    if not vfiles:
        raise FileNotFoundError('❌ فایل ویدیوی دانلود شده یافت نشد!')

    if not vfiles[0].endswith('.mp4'):
        res_cv = subprocess.run(['ffmpeg', '-y', '-i', vfiles[0], '-c', 'copy', 'input_video.mp4'], capture_output=True, text=True)
        if res_cv.returncode != 0:
            raise RuntimeError(f'❌ تبدیل ویدیو به mp4 با خطا مواجه شد:\n{res_cv.stderr}')
        os.remove(vfiles[0])

    VIDEO_FILE = 'input_video.mp4'

    print('🎵 در حال استخراج فایل صوتی جهت رونویسی دقیق با جمینای...')
    res_aud = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_FILE, '-vn', '-acodec', 'libmp3lame',
                              '-ab', '128k', '-ac', '1', 'audio.mp3'], capture_output=True, text=True)
    if res_aud.returncode != 0 or not os.path.exists('audio.mp3'):
        raise RuntimeError(f'❌ استخراج صدای ویدیو ناموفق بود:\n{res_aud.stderr}')

    AUDIO_FILE = 'audio.mp3'
    size_mb = os.path.getsize(AUDIO_FILE) / 1024 / 1024
    print(f'✅ ویدیو و صدای باکیفیت استخراج شد ({size_mb:.1f} MB)')

print('\n▶️ ویدیو و صدا آماده شدند. حالا سلول‌های بعدی را اجرا کنید.')

In [ ]:
#@title 🎙️ آپلود صدای خودتان(یا صدای دلخواه) برای نمونه
from google.colab import files
from google import genai
from google.genai import types
import os

for f in ['ref_voice.wav', 'ref_voice.mp3', 'ref_voice.m4a', 'ref_voice.ogg']:
    if os.path.exists(f):
        os.remove(f)

#@markdown یک فایل صوتی کوتاه (حدود ۱۰ ثانیه، گفتار واضح و بدون نویز زمینه) از صدای خودتان آپلود کنید.

print('📁 لطفاً فایل صوتی صدای خودتان را آپلود کنید:')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

fname = list(uploaded.keys())[0]
ext = fname.rsplit('.', 1)[-1].lower()
REF_AUDIO_PATH = f'ref_voice.{ext}'
os.rename(fname, REF_AUDIO_PATH)

MIME_MAP = {'wav': 'audio/wav', 'mp3': 'audio/mp3', 'm4a': 'audio/mp4', 'ogg': 'audio/ogg'}
ref_mime_type = MIME_MAP.get(ext, 'audio/mp3')


def transcribe_ref_audio(audio_path, mime_type):
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

    instruction = (
        "Transcribe EXACTLY the words spoken in this audio clip, in the original "
        "spoken language. Output ONLY the plain transcribed text — no timestamps, "
        "no labels, no quotation marks, no explanation, nothing else."
    )

    with open(audio_path, 'rb') as f:
        audio_bytes = f.read()

    for model_name in models:
        for attempt in range(1, 3):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=audio_bytes, mime_type=mime_type),
                        instruction
                    ]
                )
                text = response.text.strip()
                if text:
                    return text, model_name
            except Exception as e:
                print(f'    ⚠️ خطا با {model_name} (تلاش {attempt}): {e}')

    return '', None


print('🧠 در حال تشخیص متن گفته‌شده در فایل صوتی...')
REF_TEXT, used_model = transcribe_ref_audio(REF_AUDIO_PATH, ref_mime_type)

print(f'✅ صدای مرجع ذخیره شد: {REF_AUDIO_PATH}')
if REF_TEXT:
    print(f'📝 متن تشخیص داده‌شده (با {used_model}): {REF_TEXT}')
else:
    print('⚠️ متن گفتار تشخیص داده نشد؛ خود OmniVoice از Whisper برای تشخیص خودکار استفاده خواهد کرد.')


In [ ]:
#@title ##🎙️ **استخراج زیرنویس**
import os, sys, subprocess, math, time, tempfile, re
from datetime import timedelta

TARGET_MODEL = "gemini-3.5-flash-lite"
QUOTA_MARKERS = ["quota", "exceeded", "429", "resource_exhausted", "rate limit", "overloaded", "pausing"]

CUSTOM_INSTRUCTION = (
    "CRITICAL MANDATE FOR AUDIO TRANSCRIBING:\n"
    "1. LANGUAGE STRICTNESS: Transcribe EXACTLY in the ORIGINAL spoken audio language. STRICTLY DO NOT TRANSLATE into English or any other language!\n"
    "2. SUBTITLE LENGTH: Keep lines very short. Maximum 8-10 words or 35 characters per line.\n"
    "3. LINE BREAKS: Split long sentences into multiple short, timestamped blocks.\n"
    "4. HUMAN SPEECH ONLY: Transcribe ONLY actual spoken human words. COMPLETELY IGNORE non-speech sounds like [music], [applause], etc.\n"
    "5. TIMING & SYNC: Ensure timestamps match speech boundaries exactly in standard SRT format."
)

def run_transcribe(api_key, audio_path, output_path):
    script = (
        "import gemini_srt_translator as gst\n"
        f"gst.gemini_api_key = {api_key!r}\n"
        f"gst.model_name = {TARGET_MODEL!r}\n"
        f"gst.audio_file = {audio_path!r}\n"
        f"gst.output_file = {output_path!r}\n"
        f"if hasattr(gst, 'instruction'): gst.instruction = {CUSTOM_INSTRUCTION!r}\n"
        "gst.transcribe()\n"
    )
    script_path = tempfile.mktemp(suffix=".py")
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script)

    proc = subprocess.Popen(
        [sys.executable, script_path],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    quota_hit = False
    try:
        for line in proc.stdout:
            print(line, end="")
            if any(marker in line.lower() for marker in QUOTA_MARKERS):
                quota_hit = True
    finally:
        proc.wait()
        try:
            os.remove(script_path)
        except OSError:
            pass

    if quota_hit:
        return "quota"
    return "success" if proc.returncode == 0 else "error"

def parse_time(t):
    t = t.strip().replace('.', ',')
    parts = t.split(':')
    if len(parts) == 3:
        h, m, s_ms = parts
        s, ms = s_ms.split(',') if ',' in s_ms else (s_ms, '0')
        return timedelta(hours=int(h), minutes=int(m), seconds=int(s), milliseconds=int(ms[:3].ljust(3, '0')))
    elif len(parts) == 2:
        m, s_ms = parts
        s, ms = s_ms.split(',') if ',' in s_ms else (s_ms, '0')
        return timedelta(minutes=int(m), seconds=int(s), milliseconds=int(ms[:3].ljust(3, '0')))
    return timedelta(seconds=float(t.replace(',', '.')))

def format_time(td):
    total_ms = int(td.total_seconds() * 1000)
    if total_ms < 0: total_ms = 0
    h, rem = divmod(total_ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def get_duration(path):
    out = subprocess.run(
        ["ffprobe", "-i", path, "-show_entries", "format=duration", "-v", "quiet", "-of", "csv=p=0"],
        capture_output=True, text=True
    ).stdout.strip()
    return float(out) if out else 0.0

def detect_silences(path, noise_db="-30dB", min_dur=0.5):
    result = subprocess.run(
        ["ffmpeg", "-i", path, "-af", f"silencedetect=noise={noise_db}:d={min_dur}", "-f", "null", "-"],
        capture_output=True, text=True
    )
    starts = [float(x) for x in re.findall(r"silence_start:\s*([\d.]+)", result.stderr)]
    ends = [float(x) for x in re.findall(r"silence_end:\s*([\d.]+)", result.stderr)]
    return list(zip(starts, ends))

def find_cut_points(total_duration, num_chunks, silences, search_window=30):
    if num_chunks <= 1:
        return [0.0, total_duration]
    raw_targets = [total_duration * i / num_chunks for i in range(1, num_chunks)]
    cut_points = [0.0]
    for target in raw_targets:
        best = None
        best_dist = search_window
        for s, e in silences:
            mid = (s + e) / 2
            dist = abs(mid - target)
            if dist <= best_dist:
                best_dist = dist
                best = mid
        cut_points.append(best if best is not None else target)
    cut_points.append(total_duration)
    return sorted(set(cut_points))

# اطمینان از آماده بودن فایل صوتی
if ('AUDIO_FILE' not in globals() or AUDIO_FILE is None or not os.path.exists(AUDIO_FILE)):
    if 'VIDEO_FILE' in globals() and VIDEO_FILE and os.path.exists(VIDEO_FILE):
        print("🎵 در حال استخراج صدا از ویدیو...")
        subprocess.run(['ffmpeg', '-y', '-i', VIDEO_FILE, '-vn', '-acodec', 'libmp3lame', '-ab', '128k', 'audio.mp3'], capture_output=True)
        AUDIO_FILE = 'audio.mp3'

if 'AUDIO_FILE' not in globals() or AUDIO_FILE is None or not os.path.exists(AUDIO_FILE):
    print("❌ فایل صوتی یا ویدیویی معتبر برای استخراج زیرنویس پیدا نشد!")
else:
    SRT_FILE = 'audio.srt'
    chunk_dir = "audio_chunks"
    os.makedirs(chunk_dir, exist_ok=True)

    total_duration = get_duration(AUDIO_FILE)
    total_minutes = total_duration / 60
    num_chunks = 1 if total_minutes <= 15 else math.ceil(total_minutes / 15)
    print(f"⏱️ مدت فایل: {total_minutes:.1f} دقیقه → تقسیم به {num_chunks} تکه هوشمند")

    existing_chunks = sorted([f for f in os.listdir(chunk_dir) if f.endswith('.mp3')])
    if not existing_chunks:
        if num_chunks == 1:
            chunk_path = os.path.join(chunk_dir, "chunk_000.mp3")
            subprocess.run(["ffmpeg", "-i", AUDIO_FILE, "-c", "copy", chunk_path, "-y"], capture_output=True)
        else:
            print("🔍 شناسایی سکوت‌های طبیعی برای برش تمیز...")
            silences = detect_silences(AUDIO_FILE)
            cut_points = find_cut_points(total_duration, num_chunks, silences)
            print(f"✂️ برش روی {len(cut_points)-1} نقطه سکوت...")
            for i in range(len(cut_points) - 1):
                start, end = cut_points[i], cut_points[i + 1]
                out_path = os.path.join(chunk_dir, f"chunk_{i:03d}.mp3")
                subprocess.run(["ffmpeg", "-i", AUDIO_FILE, "-ss", str(start), "-to", str(end),
                                "-c:a", "libmp3lame", "-b:a", "128k", out_path, "-y"], capture_output=True)
        existing_chunks = sorted([f for f in os.listdir(chunk_dir) if f.endswith('.mp3')])
        print(f"✅ {len(existing_chunks)} تکه صوتی آماده شد.")

    durations = [get_duration(os.path.join(chunk_dir, f)) for f in existing_chunks]
    offsets = [sum(durations[:i]) for i in range(len(durations))]

    api_key = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError("❌ کلید GOOGLE_API_KEY در سلول تنظیمات وارد نشده است!")

    for i, chunk_file in enumerate(existing_chunks):
        chunk_path = os.path.join(chunk_dir, chunk_file)
        chunk_out = chunk_path.rsplit('.', 1)[0] + ".srt"

        if os.path.exists(chunk_out) and os.path.getsize(chunk_out) > 0:
            print(f"✅ تکه {i+1}/{len(existing_chunks)} از قبل آماده بود.")
            continue

        done = False
        attempt = 1
        while not done:
            print(f"🎙️ پردازش تکه {i+1}/{len(existing_chunks)} با مدل {TARGET_MODEL} (تلاش {attempt})...")
            if os.path.exists(chunk_out): os.remove(chunk_out)
            result = run_transcribe(api_key, chunk_path, chunk_out)
            if result == "success" and os.path.exists(chunk_out) and os.path.getsize(chunk_out) > 0:
                print(f"✅ تکه {i+1} آماده شد.")
                done = True
            else:
                print(f"⏳ دریافت خطا یا محدودیت سرور. ۵ ثانیه صبر و تلاش مجدد...")
                attempt += 1
                time.sleep(5)

    print("\n🔗 در حال اعمال آفست زمانی و ادغام نهایی...")
    all_entries = []
    for i, chunk_file in enumerate(existing_chunks):
        chunk_out = os.path.join(chunk_dir, chunk_file).rsplit('.', 1)[0] + ".srt"
        if not os.path.exists(chunk_out): continue
        with open(chunk_out, encoding='utf-8') as f:
            content = f.read().strip()
        if not content: continue
        blocks = re.split(r'\n\s*\n', content)
        chunk_offset = offsets[i]

        for b in blocks:
            lines = [line.strip() for line in b.strip().split('\n') if line.strip()]
            if len(lines) < 2: continue
            time_idx = -1
            for idx, l in enumerate(lines):
                if '-->' in l:
                    time_idx = idx
                    break
            if time_idx == -1: continue

            try:
                start_raw, end_raw = lines[time_idx].split('-->')
                start_td = parse_time(start_raw) + timedelta(seconds=chunk_offset)
                end_td = parse_time(end_raw) + timedelta(seconds=chunk_offset)
                timing_str = f"{format_time(start_td)} --> {format_time(end_td)}"
                dialog_text = '\n'.join(lines[time_idx+1:])
                if dialog_text:
                    all_entries.append((start_td, timing_str, dialog_text))
            except Exception:
                continue

    final_srt = []
    for idx, (_, timing, text) in enumerate(all_entries, start=1):
        final_srt.append(f"{idx}\n{timing}\n{text}")

    with open(SRT_FILE, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(final_srt) + '\n')

    SUBTITLE_READY = True
    print(f"🎉 زیرنویس دقیق با جمینای تولید شد و در {SRT_FILE} قرار گرفت.")

In [ ]:
#@title ##🌐 **ترجمه زیرنویس**
import pysrt, os, time, re, shutil, json
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types

for _f in ['dubbed_audio.mp3', 'final_dubbed_video.mp4', 'raw_segments_combined.mp3']:
    if os.path.exists(_f): os.remove(_f)
if os.path.exists('dubbing_project'): shutil.rmtree('dubbing_project')

# حداکثر کاراکتر مجاز در هر ثانیه برای جلوگیری از دویدن کلمات (استاندارد گفتار: ۱۴)
MAX_SAFE_CPS = 13.0

def translate_batch(batch_subs, target_lang, enable_summary, tone_option):
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

    items_to_send = [{"id": i + 1, "text": sub.text.strip()} for i, sub in enumerate(batch_subs)]
    json_payload = json.dumps(items_to_send, ensure_ascii=False)

    tone_prompts = {
        "عامیانه / گفتاری": "Use casual, natural everyday spoken phrasing.",
        "رسمی / اخبار / کتابی": "Use formal, precise language.",
        "آموزشی / آکادمیک": "Use clear phrasing.",
        "پرانرژی / تبلیغاتی": "Use punchy phrasing.",
        "طبیعی / پیش‌فرض": "Use natural phrasing."
    }
    tone_prompt = tone_prompts.get(tone_option, tone_prompts["طبیعی / پیش‌فرض"])

    summary_instruction = (
        "CRITICAL: Be extremely concise! Cut unnecessary words, shorten verbs, and retain ONLY the core message."
        if enable_summary else ""
    )

    prompt = (
        f"Translate the 'text' of each item into target language '{target_lang}'.\n"
        f"Tone: {tone_prompt}\n{summary_instruction}\n"
        f"Return ONLY a JSON array with matching 'id' and 'text'. No markdown.\n\n"
        f"{json_payload}"
    )

    for model_name in models:
        for attempt in range(1, 3):
            try:
                res = client.models.generate_content(
                    model=model_name, contents=prompt,
                    config=types.GenerateContentConfig(response_mime_type="application/json")
                )
                raw_json = re.sub(r'^\s*```json\s*|\s*```\s*$', '', res.text.strip(), flags=re.MULTILINE)
                data = json.loads(raw_json)
                id_to_trans = {item['id']: item['text'].strip() for item in data if 'id' in item and 'text' in item}

                result = [id_to_trans.get(i + 1, sub.text) for i, sub in enumerate(batch_subs)]
                return result
            except Exception:
                time.sleep(1.5 * attempt)

    return [sub.text for sub in batch_subs]

def translate_all(subs_list, target_lang, enable_summary, tone_option, batch_size=15):
    batches = [subs_list[i:i + batch_size] for i in range(0, len(subs_list), batch_size)]
    results = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(translate_batch, b, target_lang, enable_summary, tone_option) for b in batches]
        for future in futures:
            results.extend(future.result())
    return results

ORIGINAL_SRT_FILE = 'audio.srt'

if os.path.exists(ORIGINAL_SRT_FILE):
    subs = pysrt.open(ORIGINAL_SRT_FILE, encoding='utf-8')
    n = len(subs)

    # محاسبه بازه زمانی هر دیالوگ تا دیالوگ بعدی (به ثانیه)
    durations_sec = []
    for i in range(n):
        start_ms = (subs[i].start.hours*3600 + subs[i].start.minutes*60 + subs[i].start.seconds)*1000 + subs[i].start.milliseconds
        if i < n - 1:
            next_start_ms = (subs[i+1].start.hours*3600 + subs[i+1].start.minutes*60 + subs[i+1].start.seconds)*1000 + subs[i+1].start.milliseconds
            window_sec = (next_start_ms - start_ms) / 1000.0
        else:
            end_ms = (subs[i].end.hours*3600 + subs[i].end.minutes*60 + subs[i].end.seconds)*1000 + subs[i].end.milliseconds
            window_sec = (end_ms - start_ms) / 1000.0
        durations_sec.append(max(0.5, window_sec))

    print(f"🌐 مرحله ۱ از ۲: ترجمه کامل {n} دیالوگ...")
    translated_texts = translate_all(list(subs), Target_Language, False, Tone)

    # تشخیص ریسک بر اساس سرعت مورد نیاز برای خواندن در زمان مجاز
    risky_indices = []
    for i, trans in enumerate(translated_texts):
        cps = len(trans.strip()) / durations_sec[i]
        if cps > MAX_SAFE_CPS:
            risky_indices.append(i)

    if risky_indices:
        print(f"🔍 آنالیز زمانی: {len(risky_indices)} دیالوگ به دلیل کمبود زمان با دیالوگ بعدی تداخل خواهند داشت.")
        print("🌐 مرحله ۲ از ۲: خلاصه‌سازی فشرده برای رفع قطعی تداخل...")

        risky_subs = [subs[i] for i in risky_indices]
        risky_translated = translate_all(risky_subs, Target_Language, True, Tone)

        for idx, new_text in zip(risky_indices, risky_translated):
            # اگر ترجمه خلاصه کوتاه‌تر از ترجمه مرحله ۱ بود، جایگزین کن
            if len(new_text.strip()) < len(translated_texts[idx].strip()):
                translated_texts[idx] = new_text

    for i, t_text in enumerate(translated_texts):
        if i < len(subs): subs[i].text = t_text

    TRANSLATED_SRT = 'audio_translated.srt'
    subs.save(TRANSLATED_SRT, encoding='utf-8')
    SRT_FILE = TRANSLATED_SRT
    print(f"✅ ترجمه بهینه‌شده در {SRT_FILE} ذخیره شد.")
else:
    print("⚠️ فایل اصلی زیرنویس پیدا نشد.")

In [ ]:
#@title ##🎬 **دوبله هوشمند**
import os, subprocess, shutil
import pysrt
import torch, torchaudio
from pydub import AudioSegment
from google.colab import files as colab_files

#@markdown #### 🔊 دانلود نسخه خام هم‌زمان (بدون اعمال سرعت/سینک)
Export_Raw_Track = False #@param {type:"boolean"}

os.makedirs('dubbing_project/dubbed_segments', exist_ok=True)

STRETCH_FACTOR = 1.0


def generate_one_segment_omnivoice(i, text_to_speak, subtitle_duration, raw_path, natural_ms):
    """تولید صدای هر دیالوگ با صدای کلون‌شده‌ی خودتان (OmniVoice)"""
    if not text_to_speak or not text_to_speak.strip():
        AudioSegment.silent(duration=int(subtitle_duration * 1000)).export(raw_path, format='wav')
        natural_ms[i] = int(subtitle_duration * 1000)
        return

    for attempt in range(1, 4):
        try:
            kwargs = dict(
                text=text_to_speak,
                ref_audio=REF_AUDIO_PATH,
                num_step=int(OmniVoice_Steps),
                speed=float(OmniVoice_Speed),
            )
            if 'REF_TEXT' in globals() and REF_TEXT.strip():
                kwargs['ref_text'] = REF_TEXT

            audio = omni_model.generate(**kwargs)
            audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
            if audio_tensor.dim() == 1:
                audio_tensor = audio_tensor.unsqueeze(0)
            torchaudio.save(raw_path, audio_tensor.cpu(), 24000)

            natural_ms[i] = len(AudioSegment.from_file(raw_path))
            print(f'  ✅ [{i+1}] خوانده شد: "{text_to_speak[:40]}..."')
            return
        except Exception as e:
            if attempt == 3:
                AudioSegment.silent(duration=int(subtitle_duration * 1000)).export(raw_path, format='wav')
                natural_ms[i] = int(subtitle_duration * 1000)
                print(f'  ⚠️ [{i+1}] خطا بعد از ۳ تلاش: {e}')


def run_dubbing():
    global STRETCH_FACTOR
    STRETCH_FACTOR = 1.0

    if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
        print('❌ ابتدا سلول «آپلود صدای خودتان برای کلون» را اجرا کنید!')
        return

    srt_path = SRT_FILE if 'SRT_FILE' in globals() and SRT_FILE and os.path.exists(SRT_FILE) else 'audio_translated.srt'
    if not os.path.exists(srt_path):
        srt_path = 'audio.srt'

    if not os.path.exists(srt_path):
        print('❌ فایل زیرنویس یافت نشد!')
        return

    subs = pysrt.open(srt_path, encoding='utf-8')
    n = len(subs)

    windows_ms = [None] * n
    natural_ms = [None] * n
    raw_paths = [None] * n

    print(f'🚀 در حال گویندگی {n} دیالوگ با صدای شما (به‌صورت پیاپی روی GPU)...')

    for i, sub in enumerate(subs):
        start_ms = (sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds
        end_ms = (sub.end.hours*3600 + sub.end.minutes*60 + sub.end.seconds)*1000 + sub.end.milliseconds
        subtitle_duration = (end_ms - start_ms) / 1000.0

        if i < n - 1:
            next_sub = subs[i+1]
            next_start_ms = (next_sub.start.hours*3600 + next_sub.start.minutes*60 + next_sub.start.seconds)*1000 + next_sub.start.milliseconds
            window_ms = next_start_ms - start_ms
        else:
            window_ms = end_ms - start_ms

        raw_path = f'dubbing_project/dubbed_segments/raw_{i+1}.wav'
        windows_ms[i] = window_ms
        raw_paths[i] = raw_path

        if os.path.exists(raw_path) and os.path.getsize(raw_path) > 100:
            try:
                natural_ms[i] = len(AudioSegment.from_file(raw_path))
                print(f'  ♻️ [{i+1}/{n}] قبلاً تولید شده بود، رد شد.')
                continue
            except Exception:
                pass

        generate_one_segment_omnivoice(i, sub.text, subtitle_duration, raw_path, natural_ms)

    if any(v is None for v in natural_ms):
        print("❌ برخی از دیالوگ‌ها با خطا مواجه شدند.")
        return

    if Export_Raw_Track:
        print('🔊 در حال ساخت تراک خام (بدون اعمال سرعت)...')
        combined = AudioSegment.empty()
        for rp in raw_paths:
            if os.path.exists(rp):
                combined += AudioSegment.from_file(rp)
        raw_track_file = 'raw_segments_combined.mp3'
        combined.export(raw_track_file, format='mp3', bitrate='192k')
        colab_files.download(raw_track_file)
        print(f'✅ تراک خام ذخیره و دانلود شد: {raw_track_file}')

    idx_borrowable = list(range(n - 1)) if n > 1 else []
    total_natural = sum(natural_ms[i] for i in idx_borrowable)
    total_window  = sum(windows_ms[i] for i in idx_borrowable)
    G = (total_natural / total_window) if total_window > 0 else 1.0

    if Allow_Timeline_Stretch and G > 1.0:
        max_stretch = 1 + (Max_Stretch_Percent / 100.0)
        STRETCH_FACTOR = min(G, max_stretch)
    if STRETCH_FACTOR > 1.0:
        windows_ms = [w * STRETCH_FACTOR if i in idx_borrowable else w for i, w in enumerate(windows_ms)]

    print(f"📐 نسبت طول دوبله به زمان اصلی (G): {G:.2f}  →  صدا حدود {(G-1)*100:.0f}٪ فضا بیشتر می‌خواهد")

    R = (total_natural / total_window) if total_window > 0 else 1.0
    base_factor = min(R, 1.3) if (Balance_Speed_Across_Segments and R > 1.0) else 1.0
    hard_cap = Max_Speed_Factor if Balance_Speed_Across_Segments else 1.4

    print("⚡ در حال تنظیم زمان‌بندی و جلوگیری ۱۰۰٪ از تداخل صداها...")
    trimmed_count = 0

    for i in range(n):
        raw_path = raw_paths[i]
        final_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'

        adjusted_ms = natural_ms[i] / base_factor
        local_factor = adjusted_ms / windows_ms[i] if windows_ms[i] > 0 else 1.0
        final_factor = max(1.0, min(base_factor * max(1.0, local_factor), hard_cap))

        # ۱) اعمال سرعت هوشمند با Rubberband
        if final_factor <= 1.001:
            shutil.copy(raw_path, final_path)
        else:
            try:
                res = subprocess.run([
                    'ffmpeg', '-y', '-i', raw_path,
                    '-filter:a', f'rubberband=tempo={final_factor:.3f}',
                    final_path
                ], capture_output=True, text=True)

                if res.returncode != 0:
                    subprocess.run([
                        'ffmpeg', '-y', '-i', raw_path,
                        '-filter:a', f'atempo={final_factor:.3f}',
                        final_path
                    ], capture_output=True)
            except Exception:
                shutil.copy(raw_path, final_path)

        # ۲) گارد قطعی ضد تداخل (برش انتهای صدا قبل از دیالوگ بعدی)
        try:
            curr_audio = AudioSegment.from_file(final_path)
            # ۵۰ میلی‌ثانیه فاصله امن برای جلوگیری از چسبیدن کلمات
            safe_limit_ms = windows_ms[i] - 50

            if len(curr_audio) > safe_limit_ms and safe_limit_ms > 200:
                # برش اضافه و فید اوت نرم ۴۰ میلی‌ثانیه‌ای تا صدا تقّه نزند
                curr_audio = curr_audio[:int(safe_limit_ms)].fade_out(40)
                curr_audio.export(final_path, format='wav')
                trimmed_count += 1
        except Exception as e:
            pass

    if trimmed_count > 0:
        print(f"🛡️ گارد ضد تداخل فعال شد: انتهای {trimmed_count} دیالوگ بریده شد تا روی دیالوگ بعدی نیفتد.")

    print('✅ دوبله هوشمند با صدای شما و با همگام‌سازی کامل به پایان رسید!')

run_dubbing()

In [ ]:
#@title ##🎞️ **ترکیب نهایی و دانلود**
import subprocess, os, glob, shutil
import pysrt
from pydub import AudioSegment
from google.colab import files
from IPython.display import display, HTML

try:
    STRETCH_FACTOR
except NameError:
    STRETCH_FACTOR = 1.0

# تأخیر جبرانی ثابت برای شروع هر دیالوگ دوبله‌شده
START_DELAY_MS = 150

if not VIDEO_FILE or not os.path.exists(VIDEO_FILE):
    subs = pysrt.open(SRT_FILE, encoding='utf-8')
    total_ms = (subs[-1].end.hours*3600 + subs[-1].end.minutes*60 +
                subs[-1].end.seconds)*1000 + subs[-1].end.milliseconds
    total_ms = int(total_ms * STRETCH_FACTOR)
    final_audio = AudioSegment.silent(duration=total_ms)
    for i, sub in enumerate(subs):
        seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
        if os.path.exists(seg_path):
            start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
            seg_audio = AudioSegment.from_file(seg_path)
            final_audio = final_audio.overlay(seg_audio, position=start_ms)
    output_file = 'dubbed_audio.mp3'
    final_audio.export(output_file, format='mp3', bitrate='192k')
    files.download(output_file)
else:
    subs = pysrt.open(SRT_FILE, encoding='utf-8')
    orig_tempo = 1/STRETCH_FACTOR if STRETCH_FACTOR > 1 else 1.0

    if Keep_Original_Audio:
        filter_parts = [f'[0:a]volume={Original_Audio_Volume},atempo={orig_tempo}[orig]']
    else:
        filter_parts = ['[0:a]volume=0[orig]']

    input_args = ['-i', VIDEO_FILE]
    valid_segs = []

    for i, sub in enumerate(subs):
        seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
        if os.path.exists(seg_path):
            start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 +
                        sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
            filter_parts.append(f'[{len(valid_segs)+1}:a]adelay={start_ms}|{start_ms}[a{i+1}]')
            input_args += ['-i', seg_path]
            valid_segs.append(i+1)

    if valid_segs:
        mix_inputs = '[orig]' + ''.join(f'[a{i}]' for i in valid_segs)
        filter_parts.append(f'{mix_inputs}amix=inputs={len(valid_segs)+1}:normalize=0[aout]')

        if STRETCH_FACTOR > 1.0:
            filter_parts.append(f'[0:v]setpts={STRETCH_FACTOR}*PTS[vout]')
            video_map = ['-map', '[vout]']
            vcodec = 'libx264'
        else:
            video_map = ['-map', '0:v']
            vcodec = 'copy'

        filter_complex = ';'.join(filter_parts)
        output_file = 'final_dubbed_video.mp4'
        cmd = ['ffmpeg', '-y'] + input_args + ['-filter_complex', filter_complex] + video_map + [
            '-map', '[aout]', '-c:v', vcodec, '-c:a', 'aac', output_file
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            files.download(output_file)

image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
yt_url = 'https://youtube.com/@aigolden'
display(HTML(f'''
<div style="text-align:center;border:2px solid #e0e0e0;padding:15px;border-radius:12px;background:#f9f9f9;max-width:350px;margin:auto">
  <a href="{yt_url}" target="_blank">
    <img src="{image_url}" style="max-width:100%;border-radius:8px">
  </a>
  <p style="font-family:Vazir,sans-serif;margin-top:10px">برای آموزش‌های بیشتر ما را در یوتیوب دنبال کنید</p>
  <a href="{yt_url}" target="_blank" style="background:#FF0000;color:white;padding:10px 20px;border-radius:8px;text-decoration:none;font-weight:bold">🚀 دنبال کردن</a>
</div>
'''))
